# Update Data

### Get token
- Ga naar https://insight.quantified.eu/api/auth/login/?next=/api/doc
- Log in als JSpan@ilionx.com
- Authorize als JSpan@ilionx.com
- Ga naar Token --> POST /token/login --> Try it out --> Execute
- Copy-paste Token in de 'default_parameters.py'

In [1]:
# import requests

# # Set your username and password
# username = 'JSpan@ilionx.com'
# password = '74nK8M2Q'

# # Make a POST request to obtain the token
# token_url = 'https://insight.quantified.eu/api/token/login/'
# response = requests.post(token_url, data={'username': username, 'password': password})

# # Check if the request was successful
# if response.status_code == 200:
#     token = response.json()['access']
#     print('Token:', token)
# else:
#     print('Failed to obtain token. Status code:', response.status_code)

### Get Parameters

In [2]:
from default_parameters import get_parameters

token, device_name_ids = get_parameters()
token

'01dffd5c1dc7d5a63b06c10a63db051fb099a4690f3430c597e5a2b7cef44aab'

### Set Autorization of Quantified API

In [3]:
import requests
import pandas as pd
import datetime
import time

authorization_key = f'Bearer {token}'
authorization_key

'Bearer 01dffd5c1dc7d5a63b06c10a63db051fb099a4690f3430c597e5a2b7cef44aab'

### Call quantified API and store data in dataframe

In [4]:
df_permittivity = pd.read_excel('./data/Permittivity_updated.xlsx')
df_battery = pd.read_excel('./data/Battery_updated.xlsx')

df_sensor_id_name = pd.DataFrame.from_dict(device_name_ids, orient='index', columns=[
                                           'device_id']).reset_index().set_index('device_id')
df_sensor_id_name.columns = ['device_name']


def get_max_gateway(df):
    df_max_gateway_time = pd.DataFrame(
        df.groupby('device')['gateway_receive_time'].max())
    df_max_gateway_time.columns = ['max_gateway_receive_time']
    df_max_gateway_time.index.name = 'device_id'
    df_max_gateway_time = df_max_gateway_time.join(df_sensor_id_name)
    df_max_gateway_time.sort_values('max_gateway_receive_time')

    return df_max_gateway_time


df_max_gateway_permittivity = get_max_gateway(df_permittivity)
df_max_gateway_battery = get_max_gateway(df_battery)

df_final_permittivity = pd.DataFrame()
df_final_battery = pd.DataFrame()

resp = 200

url_list = [r"https://insight.quantified.eu/api/soil_relative_permittivity_events/",
            r"https://insight.quantified.eu/api/battery_voltage_events/"]

max_gateway_list = [df_max_gateway_permittivity, df_max_gateway_battery]

c = -1
for url_path in url_list:
    c += 1

    if resp == 401:
        print('Autorization failed')
        break

    for device_name, device_ID in device_name_ids.items():
        if resp == 401:
            break

        url = url_path

        try:
            gateway_receive_time_after = max_gateway_list[c].loc[device_ID]['max_gateway_receive_time'].strftime(
                '%Y-%m-%dT%H:%M:%SZ')
        except:
            gateway_receive_time_after = datetime(
                2020, 1, 1).strftime('%Y-%m-%dT%H:%M:%SZ') # type: ignore

        while url != None:

            headers = {
                "accept": "application/json",
                "authorization": authorization_key,
            }

            params = {
                "device_id": device_ID,
                "limit": 100,
                "gateway_receive_time_after": gateway_receive_time_after
            }

            response = requests.get(url, headers=headers, params=params)
            time.sleep(2)

            resp = response.status_code

            if resp == 200:
                df = pd.read_json(response.text)
                print(f'device: {device_ID}, count measures: {len(df)}')
                if url_path == url_list[0]:
                    df_final_permittivity = pd.concat(
                        [df_final_permittivity, df])
                elif url_path == url_list[1]:
                    df_final_battery = pd.concat([df_final_battery, df])

                txt_json = response.json()
                url = txt_json['next']
            elif resp == 401:
                break
            else:
                print(resp)
                break

df_final_permittivity

device: 93, count measures: 100
device: 93, count measures: 66
device: 54, count measures: 100
device: 54, count measures: 100
device: 54, count measures: 29
device: 193, count measures: 0
device: 208, count measures: 100
device: 208, count measures: 46
device: 356, count measures: 100
device: 356, count measures: 28
device: 360, count measures: 100
device: 360, count measures: 18
device: 361, count measures: 100
device: 361, count measures: 4
device: 362, count measures: 100
device: 362, count measures: 17
device: 369, count measures: 100
device: 369, count measures: 16
device: 395, count measures: 76
device: 372, count measures: 100
device: 372, count measures: 7
device: 400, count measures: 5
device: 401, count measures: 50
device: 396, count measures: 94
device: 397, count measures: 70
device: 399, count measures: 100
device: 399, count measures: 19
device: 404, count measures: 44
device: 405, count measures: 67
device: 408, count measures: 100
device: 408, count measures: 12
devic

,next,previous,results
0,https://insight.quantified.eu/api/soil_relativ...,NaN,"{'timestamp': 1701406028, 'gateway_receive_tim..."
1,https://insight.quantified.eu/api/soil_relativ...,NaN,"{'timestamp': 1701402036, 'gateway_receive_tim..."
2,https://insight.quantified.eu/api/soil_relativ...,NaN,"{'timestamp': 1701397871, 'gateway_receive_tim..."
3,https://insight.quantified.eu/api/soil_relativ...,NaN,"{'timestamp': 1701385894, 'gateway_receive_tim..."
4,https://insight.quantified.eu/api/soil_relativ...,NaN,"{'timestamp': 1701377737, 'gateway_receive_tim..."
...,...,...,...
76,NaN,NaN,"{'timestamp': 1699764173, 'gateway_receive_tim..."
77,NaN,NaN,"{'timestamp': 1699755972, 'gateway_receive_tim..."
78,NaN,NaN,"{'timestamp': 1699707251, 'gateway_receive_tim..."
79,NaN,NaN,"{'timestamp': 1699658673, 'gateway_receive_tim..."


In [5]:
from datetime import datetime
import pytz

print(f'length permittivity dataset old = {len(df_permittivity)}')
print(f'length battery dataset old = {len(df_battery)}')


def quantified_to_dataframe(df):
    df = pd.json_normalize(df['results'])
    df['gateway_receive_time'] = pd.to_datetime(
        df['gateway_receive_time'], utc=True)
    # replace with your local timezone
    local_tz = pytz.timezone('Europe/Amsterdam')
    df['gateway_receive_time'] = df['gateway_receive_time'].dt.tz_convert(
        local_tz)
    df['gateway_receive_time'] = df['gateway_receive_time'].dt.strftime(
        '%Y-%m-%d %H:%M:%S')
    df['gateway_receive_time'] = pd.to_datetime(df['gateway_receive_time'])

    # split datetime column into separate date and time columns
    df['date'] = df['gateway_receive_time'].dt.date
    df['time'] = df['gateway_receive_time'].dt.time

    # # extract separate columns for year, month, day, hour, minute, and second
    # df['year'] = df['gateway_receive_time'].dt.year
    # df['month'] = df['gateway_receive_time'].dt.month
    # df['day'] = df['gateway_receive_time'].dt.day
    # df['hour'] = df['gateway_receive_time'].dt.hour
    # df['minute'] = df['gateway_receive_time'].dt.minute
    # df['second'] = df['gateway_receive_time'].dt.second

    return df


df_results_permittivity = quantified_to_dataframe(df_final_permittivity)
df_results_battery = quantified_to_dataframe(df_final_battery)

df_permittivity_update = pd.concat([df_permittivity, df_results_permittivity])
df_battery_update = pd.concat([df_battery, df_results_battery])


month = str(datetime.now().month)
if len(month) == 1:
    month = '0' + month

date_str = str(datetime.now().year) + month + str(datetime.now().day)

# df_permittivity_update.to_excel(f'{date_str} - Permittivity_updated.xlsx',index=False)
# df_battery_update.to_excel(f'{date_str} - Battery_updated.xlsx',index=False)

print(f'length permittivity dataset new = {len(df_permittivity_update)}')
print(f'length battery dataset new = {len(df_battery_update)}')

df_permittivity_update.to_excel(
    './data/Permittivity_updated.xlsx', index=False)
df_battery_update.to_excel('./data/Battery_updated.xlsx', index=False)

length permittivity dataset old = 30436
length battery dataset old = 18014
length permittivity dataset new = 32790
length battery dataset new = 19267
